In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:

from rectified_flow_pytorch.rectified_flow import *
import os
# 在Python中设置环境变量（这比使用shell export更可靠）
os.environ["PATH"] = "/mnt/data/liushiyu/miniforge3/bin:" + os.environ.get("PATH", "")
# 启用多个GPU（根据实际可用GPU数量调整）
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"] = "/mnt/gptdata/liushiyu/flow_matching_pytorch_lucidrains/large_files/hf-cache"
os.environ["CUDA_HOME"] = "/mnt/gptdata/liushiyu/cuda_tools/local_cuda"

# 创建实验目录
exp_name = "exp_adni.latent"
get_ipython().system('mkdir -p /mnt/gptdata/liushiyu/flow_matching_pytorch_lucidrains/large_files/{exp_name}')
get_ipython().system('ln -sf ./{exp_name} /mnt/gptdata/liushiyu/flow_matching_pytorch_lucidrains/large_files/{exp_name}')

# 显示NVIDIA系统管理界面
get_ipython().system('nvidia-smi')

# 验证CUDA_HOME设置
print(f"CUDA_HOME环境变量: {os.environ.get('CUDA_HOME', '未设置')}")
if os.path.exists(os.environ.get('CUDA_HOME', '')):
    print("CUDA_HOME路径存在")
    get_ipython().system('ls $CUDA_HOME')
else:
    print("警告: CUDA_HOME路径不存在!")

# 验证CUDA是否可用
import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA设备数量: {torch.cuda.device_count()}")
    print(f"当前CUDA设备: {torch.cuda.current_device()}")
    print(f"设备名称: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    # 验证CUDA_VISIBLE_DEVICES设置
    print(f"CUDA_VISIBLE_DEVICES环境变量: {os.environ.get('CUDA_VISIBLE_DEVICES', '未设置')}")


import glob 
import os 
dataset_path ="/mnt/gptdata/liushiyu/datasets"
target_path = "/mnt/gptdata/liushiyu/latents/adni"
os.makedirs(target_path,exist_ok=True)
all_nifti= sorted(glob.glob(os.path.join(dataset_path,"*nii*")))
all_nifti=[{"image":nifti} for nifti in all_nifti]
from shiyu_utils.maisi_transforms import VAE_Transform
transform = VAE_Transform(is_train=False,random_aug=False,val_patch_size=(224,224,224),spacing_type="fixed",spacing=(1.,1.,1.))
transform = transform.transform_dict["mri"]
def test_transform():
    data = transform(all_nifti[0])["image"]
    print(data.shape)
    from shiyu_utils.plot_3d_data import plot_3d_data
    plot_3d_data(data)

import pandas as pd 
df = pd.read_csv("3.adni.cfm.merged.csv")[["DX_bl","path"]]
data_list = df.to_dict(orient="records")
data_list_new =[]
label_map = {"CN":1,"AD":3,"EMCI":2,"LMCI":2}
for data in data_list:
    data_list_new.append({
        "label":label_map[data["DX_bl"]],
        #"image_old":data["path"],
        "image":data["path"].replace(dataset_path,target_path),
    })
data_list_new[:3]


# In[3]:


import monai.data as md 
from tqdm.notebook import tqdm 
scale=32
shift=128
forward_func = lambda x: (x*scale+shift).clamp(0,255).to(torch.uint8)
backward_func = lambda x: (x.float()-shift)/scale


# In[4]:


import torch 
import monai

from torch.utils.data import Dataset
import glob 
import os 
import torch 
def load_func(x):
    x = torch.load(x,weights_only=False)[0]
    x = (x.float()-shift)/scale
    x = x *0.25
    return x
import monai.data as md  

class ADNILatentConditionDataset(Dataset):
    def __init__(self,files,train=True):
        self.files=files
        self.train=train

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):
        latent = load_func(self.files[idx]["image"])
        label = self.files[idx]["label"]
        # randomly 15% map label to zero 
        if self.train and torch.rand(1).item()<0.15:
            label = 0
        return latent,label


# In[5]:


from errno import ESTALE
from rectified_flow_pytorch import RectifiedFlow, Unet, Trainer
from monai.networks.nets import DiffusionModelUNet
from torch import nn, pi, cat, stack, from_numpy
from einops import rearrange
class monai_wrapper(torch.nn.Module):
    def __init__(self,model,mean_variance_net=False):
        super().__init__()
        self.model = model
        self.mean_variance_net = mean_variance_net

    def forward(self,x,times,cond=None):
        context = cond 
        timesteps = ((1.- times) * 1000 ).long()
        out= self.model(x,context=context,timesteps=timesteps)
        if self.mean_variance_net:
            mean, log_var = rearrange(out, 'b (c mean_log_var) h w d -> mean_log_var b c h w d', mean_log_var = 2)
            variance = log_var.exp() # variance needs to be positive
            return stack((mean,variance))
        else: 
            return out 


# In[6]:


from rectified_flow_pytorch import RectifiedFlow, Unet, Trainer

dataset = ADNILatentConditionDataset(data_list_new)

monai_model = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    num_res_blocks=(2,2,2),
    channels=(64,64,128),
    attention_levels=(False,False,True),
    norm_num_groups=32,
    num_head_channels=64,
    cross_attention_dim=False,
    with_conditioning=None,
    use_flash_attention=True,
    )

model = monai_wrapper(monai_model,mean_variance_net=False)
rectified_flow = RectifiedFlow(model,mean_variance_net=False,data_shape=(4,64,64,64))


Thu Sep 11 08:40:41 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA TITAN RTX               On  | 00000000:04:00.0 Off |                  N/A |
| 40%   41C    P8              16W / 280W |      3MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:

from torch.optim import Adam
from accelerate import Accelerator
from torch.utils.data import DataLoader
from ema_pytorch import EMA

def cycle(dl):
    while True:
        for batch in dl:
            yield batch

from monai.bundle import ConfigParser
config = ConfigParser()
config.read_config('shiyu_utils/config_maisi3d-rflow.json')
config["autoencoder_def"]["num_splits"]=1
autoencoder =config.get_parsed_content('autoencoder_def',instanitiate=True)
autoencoder.load_state_dict(torch.load("models/autoencoder_epoch273.pt"))
autoencoder.eval()
from tqdm import tqdm 
from rectified_flow_pytorch import Trainer
class MyTrainer(Trainer):
    def _save_2d_in_png(self,sampled,fname):
        sampled = rearrange(sampled, '(row col) c h w -> c (row h) (col w)', row = self.num_sample_rows)
        sampled.clamp_(0., 1.)

        save_image(sampled, fname)
        return sampled

    def _save_3d_in_png(self,data,fname):
        _,_,h,w,d = data.shape
        hh,ww,dd = h//2,w//2,d//2
        data_xy=data[:,:,hh,:,:]
        data_yz=data[:,:,:,ww,:]
        data_xz=data[:,:,:,:,dd]
        fname_xy=fname.replace(".png","_xy.png")
        fname_yz=fname.replace(".png","_yz.png")
        fname_xz=fname.replace(".png","_xz.png")
        sampled_xy= self._save_2d_in_png(data_xy,fname_xy)
        sampled_yz= self._save_2d_in_png(data_yz,fname_yz)
        sampled_xz= self._save_2d_in_png(data_xz,fname_xz)
        return sampled_xy,sampled_yz,sampled_xz

    def _process_input(self,data):
        noise,cond=None,None
        if isinstance(data,tuple) or isinstance(data,list):
            data,cond=data[0],data[1]
        elif isinstance(data,dict):
            data,noise=data["image"],data["source_image"]
        return data,noise,cond
    
    def sample(self, fname):
        eval_model = default(self.ema_model, self.model)
        dl = cycle(self.dl)
        mock_data = next(dl)
        mock_data,mock_noise,mock_cond =self._process_input(mock_data)
        data_shape = mock_data.shape[1:]

        additional_sample_kwargs = dict()
        if isinstance(eval_model.model, RectifiedFlow):
            additional_sample_kwargs.update(temperature = self.sample_temperature)
            additional_sample_kwargs.update(noise = mock_noise)
            additional_sample_kwargs.update(cond = mock_cond)

        with torch.no_grad():
            sampled = eval_model.sample(
                batch_size = self.num_samples,
                data_shape = data_shape,
                **additional_sample_kwargs
            )
            global autoencoder
            autoencoder = autoencoder.to(trainer.accelerator.device)
            sampled_collect=[]
            with torch.cuda.amp.autocast(True):
                for sample_per_batch in tqdm(sampled):
                    sample_per_batch = autoencoder.decode_stage_2_outputs(sample_per_batch[None,...]/0.25)
                    sampled_collect.append(sample_per_batch)
                sampled = torch.cat(sampled_collect,dim=0)
        if len(self.model.data_shape)>3:
            return self._save_3d_in_png(sampled,fname)
            # sampled = rearrange(sampled, '(row col) c h w -> c (row h) (col w)', row = self.num_sample_rows)
            # sampled.clamp_(0., 1.)

            # save_image(sampled, fname)
            # return sampled
        else:
            return self._save_2d_in_png(sampled,fname)
    def forward(self):

        dl = cycle(self.dl)

        for ind in range(self.num_train_steps):
            step = ind + 1

            self.model.train()

            data = next(dl)

            data,noise,cond=self._process_input(data)
            if self.return_loss_breakdown:
                loss, loss_breakdown = self.model(data,noise=noise,cond=cond, return_loss_breakdown = True)
                self.log(loss_breakdown._asdict(), step = step)
            else:
                loss = self.model(data)

            self.accelerator.print(f'[{step}] loss: {loss.item():.3f}')
            self.accelerator.backward(loss)

            self.accelerator.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

            self.optimizer.step()
            self.optimizer.zero_grad()

            if getattr(self.model, 'use_consistency', False):
                self.model.ema_model.update()

            if self.is_main and self.use_ema:
                self.ema_model.ema_model.data_shape = self.model.data_shape
                self.ema_model.update()

            self.accelerator.wait_for_everyone()
            if self.is_main:

                if divisible_by(step, self.save_results_every) or step==1: # we want to sample for the first step to debug sample 

                    sampled = self.sample(fname = str(self.results_folder / f'results.{step}.png'))

                    self.log_images(sampled, step = step)

                if divisible_by(step, self.checkpoint_every) or step==1:
                    self.save(f'checkpoint.{step}.pt')

            self.accelerator.wait_for_everyone()

        print('training complete')
trainer = MyTrainer(
    rectified_flow,
    dataset = dataset,
    batch_size=4,
    num_train_steps = 70_000,
    sample_temperature = 1.5,
    checkpoint_every=2000,
    results_folder = './results'   # samples will be saved periodically to this folder
)

trainer()

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[1] loss: 1.061


/tmp/ipykernel_63805/3380897858.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):
100%|██████████| 16/16 [00:33<00:00,  2.09s/it]


[2] loss: 0.977
[3] loss: 1.249
[4] loss: 1.013
[5] loss: 1.003
[6] loss: 0.745
[7] loss: 1.069
[8] loss: 0.816
[9] loss: 0.827
[10] loss: 0.954
[11] loss: 1.132
[12] loss: 1.083
[13] loss: 0.908
[14] loss: 0.924
[15] loss: 1.032
[16] loss: 0.904
[17] loss: 1.042
[18] loss: 0.921
[19] loss: 0.729
[20] loss: 0.705
[21] loss: 0.875
[22] loss: 0.884
[23] loss: 0.812
[24] loss: 0.970
[25] loss: 0.824
[26] loss: 0.889
[27] loss: 0.626
[28] loss: 0.711
[29] loss: 0.938
[30] loss: 0.911
[31] loss: 0.666
[32] loss: 0.670
[33] loss: 0.862
[34] loss: 0.472
[35] loss: 0.671
[36] loss: 0.608
[37] loss: 0.823
[38] loss: 0.673
[39] loss: 0.845
[40] loss: 0.673
[41] loss: 0.915
[42] loss: 0.758
[43] loss: 0.784
[44] loss: 0.939
[45] loss: 0.611
[46] loss: 0.909
[47] loss: 0.393
[48] loss: 0.831
[49] loss: 0.572
[50] loss: 0.652
[51] loss: 0.608
[52] loss: 0.644
[53] loss: 1.087
[54] loss: 0.594
[55] loss: 0.976
[56] loss: 0.681
[57] loss: 0.579
[58] loss: 1.044
[59] loss: 0.657
[60] loss: 0.842
[61] 

100%|██████████| 16/16 [00:33<00:00,  2.11s/it]


...

100%|██████████| 16/16 [00:33<00:00,  2.11s/it]


[801] loss: 0.996
[802] loss: 0.872
[803] loss: 0.835
[804] loss: 0.768
[805] loss: 0.742
[806] loss: 0.658
[807] loss: 0.904
[808] loss: 0.307
[809] loss: 0.664
[810] loss: 0.687
[811] loss: 1.019
[812] loss: 0.944
[813] loss: 0.895
[814] loss: 0.916
[815] loss: 0.592
[816] loss: 0.986
[817] loss: 0.985
[818] loss: 0.845
[819] loss: 0.603
[820] loss: 0.625
[821] loss: 0.915
[822] loss: 0.940
[823] loss: 0.768
[824] loss: 0.826
[825] loss: 0.841
[826] loss: 0.703
[827] loss: 0.844
[828] loss: 1.001
[829] loss: 0.897
[830] loss: 0.675
[831] loss: 1.121
[832] loss: 0.748
[833] loss: 0.892
[834] loss: 0.879
[835] loss: 0.599
[836] loss: 0.782
[837] loss: 0.933
[838] loss: 0.821
[839] loss: 0.747
[840] loss: 0.810
[841] loss: 0.780
[842] loss: 0.896
[843] loss: 1.036
[844] loss: 0.569
[845] loss: 0.869
[846] loss: 0.783
[847] loss: 0.607
[848] loss: 1.073
[849] loss: 0.624
[850] loss: 0.709
[851] loss: 0.800
[852] loss: 0.940
[853] loss: 0.786
[854] loss: 0.727
[855] loss: 0.580
[856] loss

100%|██████████| 16/16 [00:33<00:00,  2.11s/it]


[901] loss: 0.830
[902] loss: 0.565
[903] loss: 0.932
[904] loss: 0.954
[905] loss: 0.414
[906] loss: 0.866
[907] loss: 0.701
[908] loss: 0.867
[909] loss: 0.845
[910] loss: 0.694
[911] loss: 0.655
[912] loss: 0.823
[913] loss: 1.077
[914] loss: 1.043
[915] loss: 0.760
[916] loss: 0.566
[917] loss: 0.628
[918] loss: 0.703
[919] loss: 0.872
[920] loss: 0.582
[921] loss: 0.724
[922] loss: 0.728
[923] loss: 0.973
[924] loss: 0.609
[925] loss: 0.677
[926] loss: 0.843
[927] loss: 0.844
[928] loss: 0.652
[929] loss: 0.879
[930] loss: 0.911
[931] loss: 0.952
[932] loss: 1.027
[933] loss: 0.585
[934] loss: 0.725
[935] loss: 0.840
[936] loss: 0.747
[937] loss: 0.591
[938] loss: 0.706
[939] loss: 1.038
[940] loss: 0.723
[941] loss: 0.717
[942] loss: 0.790
[943] loss: 0.983
[944] loss: 0.749
[945] loss: 0.611
[946] loss: 0.998
[947] loss: 0.828
[948] loss: 0.580
[949] loss: 0.830
[950] loss: 0.852
[951] loss: 0.613
[952] loss: 0.815
[953] loss: 0.807
[954] loss: 0.526
[955] loss: 0.979
[956] loss

 50%|█████     | 8/16 [00:16<00:16,  2.10s/it]